In [ ]:
import multiprocessing
import sys
from concurrent.futures import ALL_COMPLETED, Future, wait
from datetime import datetime, time
from logging import DEBUG, INFO
from pathlib import Path
from time import sleep

from vnpy.trader.setting import SETTINGS
from vnpy.trader.engine import MainEngine, EventEngine, OmsEngine
from vnpy.trader.object import OrderData, TradeData, PositionData, AccountData
# from vnpy_tts import TtsGateway
from vnpy_ctp import CtpGateway
# from vnpy_ctptest import CtptestGateway
from vnpy_simplestrategy import StrategyEngine, SimpleStrategyApp
from vnpy_ctastrategy import CtaEngine, CtaStrategyApp

# 紫金天风 实盘
ctp_setting = {
    "用户名": "89110038",
    "密码": "wjzb7777",
    "经纪商代码": "0001",
    "交易服务器": "101.230.80.85:41206",
    "行情服务器": "101.230.80.85:41214",
    "产品名称": "client_unboundream_v2",
    "授权编码": "3J474CT8DL4EUW6F",
    "产品信息": "unboundream"
}

# Chinese futures market trading period (day/night)
DAY_START = time(8, 45)
DAY_END = time(15, 0)

NIGHT_START = time(20, 45)
NIGHT_END = time(2, 45)


def check_trading_period() -> bool:
    """"""
    current_time = datetime.now().time()

    trading = False
    if (
        (current_time >= DAY_START and current_time <= DAY_END)
        or (current_time >= NIGHT_START)
        or (current_time <= NIGHT_END)
    ):
        trading = True

    # return True  # 适配 7*24
    return trading


# --- 构建主引擎 ---

# 创建主引擎
main_engine: MainEngine = MainEngine()

# 获取订单引擎
oms_engine: OmsEngine = main_engine.get_engine("oms") # type: ignore

# 使用 CtpGateway
main_engine.add_gateway(CtpGateway)
main_gateway: CtpGateway = main_engine.get_gateway("CTP") # type: ignore

# --- 登录 CTP ---

# 尝试登录，如果失败则退出程序
main_gateway.connect(ctp_setting)
main_engine.write_log("连接CTP接口")

tries = 0
while not main_gateway.td_api.login_status and tries < 10:
    tries += 1
    main_engine.write_log("等待CTP接口登录")
    sleep(1)
if not main_gateway.td_api.login_status:
    main_engine.write_log("CTP接口登录失败")
    sys.exit(0)

tries = 0
while not main_gateway.td_api.contract_inited and tries < 180:
    tries += 1
    main_engine.write_log("等待CTP合约信息初始化")
    sleep(1)
if not main_gateway.td_api.contract_inited:
    main_engine.write_log("CTP接口合约初始化失败")
    sys.exit(0)

main_engine.write_log("主引擎创建成功")

main_engine.get_all_contracts()

TRADER_DIR: c:\vnpy_all\udt\udt
TEMP_DIR: c:\vnpy_all\udt\udt\data
2025-06-18 13:52:10.517 | INFO |  | 连接CTP接口
2025-06-18 13:52:10.519 | INFO |  | 等待CTP接口登录
2025-06-18 13:52:10.592 | INFO | CTP | 行情服务器连接成功
2025-06-18 13:52:10.621 | INFO | CTP | 行情服务器登录成功
2025-06-18 13:52:10.622 | INFO | CTP | 交易服务器连接成功
2025-06-18 13:52:10.783 | INFO | CTP | 交易服务器授权验证成功
2025-06-18 13:52:10.809 | INFO | CTP | 交易服务器登录成功
2025-06-18 13:52:11.132 | INFO | CTP | 结算信息确认成功
2025-06-18 13:52:11.525 | INFO |  | 等待CTP合约信息初始化
2025-06-18 13:52:12.588 | INFO |  | 等待CTP合约信息初始化
2025-06-18 13:52:13.628 | INFO |  | 等待CTP合约信息初始化
2025-06-18 13:52:14.695 | INFO |  | 等待CTP合约信息初始化
2025-06-18 13:52:15.696 | INFO |  | 等待CTP合约信息初始化
2025-06-18 13:52:15.952 | INFO | CTP | 合约信息查询成功
2025-06-18 13:52:16.708 | INFO |  | 主引擎创建成功


[ContractData(gateway_name='CTP', extra=None, symbol='MA512', exchange=<Exchange.CZCE: 'CZCE'>, name='MA512', product=<Product.FUTURES: '期货'>, size=10, pricetick=1.0, min_volume=1, max_volume=1000, stop_supported=False, net_position=False, history_data=False, option_strike=None, option_underlying=None, option_type=None, option_listed=None, option_expiry=None, option_portfolio=None, option_index=None),
 ContractData(gateway_name='CTP', extra=None, symbol='CF601', exchange=<Exchange.CZCE: 'CZCE'>, name='CF601', product=<Product.FUTURES: '期货'>, size=5, pricetick=5.0, min_volume=1, max_volume=1000, stop_supported=False, net_position=False, history_data=False, option_strike=None, option_underlying=None, option_type=None, option_listed=None, option_expiry=None, option_portfolio=None, option_index=None),
 ContractData(gateway_name='CTP', extra=None, symbol='FG601', exchange=<Exchange.CZCE: 'CZCE'>, name='FG601', product=<Product.FUTURES: '期货'>, size=20, pricetick=1.0, min_volume=1, max_volume

2025-06-18 16:28:07.552 | INFO | CTP | 交易服务器连接断开，原因 4097
2025-06-18 16:28:07.565 | INFO | CTP | 行情服务器连接断开，原因 4097
2025-06-18 16:55:52.872 | INFO | CTP | 交易服务器连接断开，原因 4097
2025-06-18 16:55:52.987 | INFO | CTP | 交易服务器连接成功
2025-06-18 16:55:53.048 | INFO | CTP | 交易服务器授权验证失败，代码：7，信息：CTP:还没有初始化
2025-06-18 16:55:53.713 | INFO | CTP | 行情服务器连接成功
2025-06-18 16:55:53.747 | INFO | CTP | 行情服务器登录成功
2025-06-18 16:56:25.425 | INFO | CTP | 交易服务器连接断开，原因 4097
2025-06-18 16:56:25.435 | INFO | CTP | 行情服务器连接断开，原因 4097
2025-06-18 16:56:25.569 | INFO | CTP | 交易服务器连接成功
2025-06-18 16:56:25.601 | INFO | CTP | 行情服务器连接成功
2025-06-18 16:56:25.625 | INFO | CTP | 行情服务器登录成功
2025-06-18 16:56:48.947 | INFO | CTP | 交易服务器授权验证失败，代码：7，信息：CTP:还没有初始化
2025-06-18 17:11:14.620 | INFO | CTP | 交易服务器连接断开，原因 4097
2025-06-18 17:11:14.806 | INFO | CTP | 行情服务器连接断开，原因 4097
2025-06-18 19:21:54.949 | INFO | CTP | 行情服务器连接成功
2025-06-18 19:21:54.986 | INFO | CTP | 行情服务器登录成功
2025-06-18 19:21:55.220 | INFO | CTP | 交易服务器连接成功
2025-06-18 19:21:55.

In [6]:
import pandas as pd
import akshare as ak

In [4]:
df = pd.DataFrame(main_engine.get_all_contracts())

In [7]:
tool_trade_date_hist_sina_df = ak.tool_trade_date_hist_sina()
tool_trade_date_hist_sina_df['trade_date'] = pd.to_datetime(tool_trade_date_hist_sina_df['trade_date'])
current_time = datetime.now()
current_date = pd.to_datetime(current_time.strftime("%Y-%m-%d"))
trade_date = pd.to_datetime(current_time.strftime("%Y-%m-%d"))

matching_index = tool_trade_date_hist_sina_df.index[tool_trade_date_hist_sina_df['trade_date'] == trade_date]

previous_dates = tool_trade_date_hist_sina_df.loc[matching_index[0] - 2:matching_index[0] - 1, 'trade_date'].dt.strftime("%Y%m%d").values
by_day, y_day = previous_dates[0], previous_dates[1]

In [10]:
by_data = ak.get_futures_daily(start_date=by_day, end_date=by_day, market='INE')

In [11]:
by_data

,symbol,date,open,high,low,close,volume,open_interest,turnover,settle,pre_settle,variety
0,SC2507,20250616,539.4,553.6,532.5,541.6,228321,16919,12351624.84,540.9,513.7,SC
1,SC2508,20250616,530.4,536.7,520.1,532.9,131524,29864,6960533.8,529.2,514.4,SC
2,SC2509,20250616,520,528.9,512.1,524.6,30709,12529,1599073.3,520.7,512.9,SC
3,SC2510,20250616,517.7,521.4,505.8,516.7,5562,2283,286106.74,514.3,510.4,SC
4,SC2511,20250616,514.1,518.8,501.4,511.9,730,469,37237.39,510.1,511.1,SC
...,...,...,...,...,...,...,...,...,...,...,...,...
61,EC2604,20250616,1300.1,1302.3,1250,1273.7,873,3560,5570.25,1276.1,1296.3,EC
62,SC_TAS2507,20250616,,,,0,,,,,,SC_TAS
63,SC_TAS2508,20250616,,,,0,,,,,,SC_TAS
64,SC_TAS2509,20250616,,,,0,,,,,,SC_TAS
